# Module 2.4: Interactive Telemetry Plotting with Matplotlib & Plotly
### 🇵🇱 Interaktywna Wizualizacja Telemetrii z Matplotlib i Plotly

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/max480425/industrial-ml-course/blob/main/public/notebooks/m02_l04_interactive_telemetry_plotting.ipynb)

**Industrial Machine Learning & Predictive Maintenance Interactive Laboratory**

> **Summary:** Build synchronized multi-channel dashboards for machine telemetry, overlay threshold alarms, and enable sub-second zoom into high-speed events.  
> 🇵🇱 **Streszczenie:** Twórz zsynchronizowane pulpity telemetryczne maszyn, nakładaj progi alarmowe i badaj szybkie zdarzenia awaryjne w powiększeniu.

- **Estimated Time:** 25 minutes
- **Runtime:** Python 3.10+ (CPU or Free Tier T4 GPU)
- **Packages:** `numpy`, `scipy`, `scikit-learn`, `pandas`, `matplotlib`, `seaborn`
---


In [ ]:
# Step 1: Install & Verify Python Scientific Environment
!pip install -q numpy scipy scikit-learn pandas matplotlib seaborn

import numpy as np
import pandas as pd
import scipy.signal as signal
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'monospace'
print("✓ Industrial ML environment loaded successfully.")


### Key Engineering Concepts:
1. **Multi-Physics Time-Series**: Machines combine incompatible physical units (Temperature in °C, Pressure in bar, Vibration in m/s²). Plotting on a single axis obscures dynamics.
2. **Dual & Triple Y-Axes**: Synchronizing multiple vertical scales enables direct visual correlation between pressure surges and temperature spikes.
3. **Interactive Inspection**: Static PNG charts prevent drill-down into transient micro-stoppages. Interactive Plotly charts allow maintenance teams to pan, zoom, and inspect exact PLC timestamps.


In [ ]:
# 1. Synthesize Synchronized Multi-Physics Factory Incident
np.random.seed(42)
n_pts = 600
time_axis = pd.date_range("2026-06-01 08:00:00", periods=n_pts, freq="1s")

# Hydraulic Pressure (bar): Normal 160 bar, drops during cavitation event
pressure = 160.0 + np.random.normal(0, 2.5, n_pts)
pressure[300:380] -= np.linspace(0, 45, 80)
pressure[380:450] -= 45.0
pressure[450:520] += np.linspace(-45, 0, 70)

# Motor Temperature (°C): Overheating consequence
temp = 42.0 + np.cumsum(np.where(pressure < 130.0, 0.25, -0.05)) + np.random.normal(0, 0.3, n_pts)

df_incident = pd.DataFrame({"timestamp": time_axis, "pressure_bar": pressure, "temp_c": temp})
print("✓ Incident dataset generated: cavitation and thermal runaway.")


In [ ]:
# 2. Dual-Axis Production Telemetry Dashboard
fig, ax1 = plt.subplots(figsize=(14, 5))

color_p = '#0284c7'
ax1.set_xlabel("Time [HH:MM:SS]", fontweight='bold')
ax1.set_ylabel("Hydraulic Pressure [bar]", color=color_p, fontweight='bold')
line1 = ax1.plot(df_incident["timestamp"], df_incident["pressure_bar"], color=color_p, linewidth=1.8, label='Pressure (bar)')
ax1.tick_params(axis='y', labelcolor=color_p)
ax1.axhline(130.0, color='#ef4444', linestyle='--', alpha=0.7, label='Cavitation Alarm Level (130 bar)')

# Twin axis for Temperature
ax2 = ax1.twinx()
color_t = '#ef4444'
ax2.set_ylabel("Motor Winding Temp [°C]", color=color_t, fontweight='bold')
line2 = ax2.plot(df_incident["timestamp"], df_incident["temp_c"], color=color_t, linewidth=2.0, linestyle='-', label='Temperature (°C)')
ax2.tick_params(axis='y', labelcolor=color_t)

# Shaded fault zone
ax1.axvspan(df_incident["timestamp"].iloc[300], df_incident["timestamp"].iloc[520], color='#ef4444', alpha=0.15, label='Cavitation Window')

plt.title("Synchronized Multi-Physics Telemetry: Hydraulic Cavitation & Overheating", fontsize=13, fontweight='bold')
ax1.grid(True, linestyle='--', alpha=0.3)
fig.tight_layout()
plt.show()


## 📋 Practical Hands-On Checklist

- [ ] **Task 1:** Construct a multi-panel synchronized time-series figure in Matplotlib *(🇵🇱 Zbuduj wielopanelowy, zsynchronizowany w czasie wykres w Matplotlib)*
- [ ] **Task 2:** Overlay ISO 10816 machinery vibration severity limit bands (Green, Yellow, Red) *(🇵🇱 Nałóż strefy alarmowe normy ISO 10816 (Zielona, Żółta, Czerwona))*
- [ ] **Task 3:** Render an interactive zoomable Plotly telemetry dashboard for plant operators *(🇵🇱 Wygeneruj interaktywny wykres Plotly z płynnym zoomem dla operatorów linii)*


In [ ]:
# Exercise Challenge:
# Identify the maximum temperature reached during the cavitation window
max_temp_cavitation = df_incident.loc[300:520, "temp_c"].max()
print(f"Peak Cavitation Temperature: {max_temp_cavitation:.2f}°C")
assert max_temp_cavitation > 55.0, "Expected temperature surge during cavitation."
print("✓ Multi-physics telemetry visualization test passed.")
